# 4. Capturing and Storing a League

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `03_sleeper_api.ipynb` first if you haven't.

`03_sleeper_api.ipynb` called `SleeperClient` directly for one-off lookups. This notebook builds a *complete, reproducible* dataset for a league: an immutable raw snapshot, full multi-season history, standings and playoff results, weekly matchups, transactions, and roster composition -- all persisted to disk so the rest of this tutorial (and your own analysis) can query them without hitting Sleeper again. Mirrors the [Capturing and Storing a League](https://nolmacdonald.github.io/nuclearff/tutorial/04_capturing_a_league.html) docs page.

Everything below writes into one shared DuckDB database (`<cache_dir>/nuclearff.duckdb`) and one raw-snapshot directory. This is the notebook every later chapter in this series builds on -- run it before any chapter from here on.

## A league snapshot

`fetch_league_snapshot` fetches every read-only endpoint that describes a league -- settings, users, rosters, drafts -- into one `LeagueSnapshot`, and `write_snapshot` writes it to an immutable, timestamped directory (never overwritten -- a second run creates a new timestamp):

In [1]:
from nuclearff.sleeper import SleeperClient, fetch_league_snapshot, write_snapshot

league_id = "1367225133634191360"
raw_dir = "./demo/data/raw"

with SleeperClient(cache_dir="./demo/data/cache") as client:
    snapshot = fetch_league_snapshot(client, league_id)
    target = write_snapshot(snapshot, raw_dir)

print(f"League:   {snapshot.league_name} ({snapshot.metadata.league_id})")
print(
    f"Season:   {snapshot.league.get('season')} status={snapshot.league.get('status')}"
)
print(f"Teams:    {snapshot.league.get('total_rosters')}")
print(f"Snapshot: {target}")

League:   NUCLEARFF REDRAFT (1367225133634191360)
Season:   2026 status=in_season
Teams:    10
Snapshot: demo/data/raw/sleeper/1367225133634191360/20260922T104953Z


A snapshot also carries `anomalies` -- league settings that are internally inconsistent or otherwise worth a second look before trusting them for valuation, from `detect_anomalies`:

In [2]:
for anomaly in snapshot.anomalies:
    print(f"[{anomaly.severity.upper()}] {anomaly.code}: {anomaly.message}")
    print(f"  -> {anomaly.action}")

[WARNING] draft_rounds_mismatch: League settings report draft_rounds=3 but the draft object reports rounds=15.
  -> Trust the draft object. The league field is stale; use 15 rounds for pick-gap and VONA math.
[WARNING] keepers_in_redraft: League type is 0 (redraft) but max_keepers=1.
  -> Treat the league as redraft, but confirm in the Sleeper UI whether a keeper is actually in play.
[INFO] median_scoring: Each team also plays the weekly league median (league_average_match=1).
  -> Weight weekly floor more heavily than ceiling when breaking ties between players.
[INFO] no_kicker_or_defense: No kicker, defense, or IDP roster slots.
  -> Kicker and defensive scoring keys in the payload are inert for lineup construction.


Real, contradictory settings like a stale `draft_rounds` field or a keeper cap set on a nominally-redraft league are exactly why `fetch_league_snapshot` surfaces them up front rather than letting a downstream valuation quietly use the wrong number.

> **Why this matters:** it's tempting to skip reading `anomalies` when the snapshot call otherwise "worked." Don't -- a WARNING-severity anomaly here is exactly the kind of thing that silently wrecks a valuation three chapters later (e.g. drafting against `draft_rounds=3` when the real draft ran 15 rounds) with no exception anywhere to point at the cause. Read this list every time you capture a league you haven't seen before.

## Multi-season history

Sleeper links a league to its prior season via `previous_league_id`. `walk_league_chain` walks that chain all the way back, and `write_league_tables` persists it to DuckDB:

In [3]:
from nuclearff.sleeper import walk_league_chain, write_league_tables

db_path = "./demo/data/cache/nuclearff.duckdb"

with SleeperClient(cache_dir="./demo/data/cache") as client:
    leagues = walk_league_chain(client, league_id, max_seasons=20)
    raw_count, config_count = write_league_tables(leagues, db_path)

print(f"History:  {len(leagues)} season(s) walked")
print(f"Configs:  {config_count}/{raw_count} parsed into LeagueConfig")

History:  6 season(s) walked
Configs:  6/6 parsed into LeagueConfig


For this league, that reaches all the way back to its 2021 inception -- six real seasons, not a synthetic sample. Two tables land in DuckDB: `sleeper_leagues` (the raw settings/scoring/roster JSON per season) and `sleeper_league_configs` (the same data, typed and flattened via `LeagueConfig`). A season that fails to parse keeps its raw row and just skips the typed one, rather than losing the whole hop; the walk itself stops cleanly at the end of the chain, not with an error.

Every function below takes that same `leagues` list -- resolve the chain once, then fetch as many kinds of data against it as you want.

## Standings and playoff results

`fetch_and_write_standings` computes, per season, each roster's regular-season record *and* where they actually finished after the playoffs:

In [4]:
from nuclearff.sleeper import fetch_and_write_standings

with SleeperClient(cache_dir="./demo/data/cache") as client:
    standings_count, matches_count = fetch_and_write_standings(client, leagues, db_path)

print(f"Standings: {standings_count} roster-season(s)")
print(f"Playoffs:  {matches_count} bracket match(es)")

Standings: 60 roster-season(s)
Playoffs:  66 bracket match(es)


What makes this worth having is that regular-season record and final standing are genuinely different things. Query both columns for this league's completed 2025 season:

In [5]:
import duckdb

with duckdb.connect(db_path, read_only=True) as conn:
    season_2025_rows = conn.execute(
        """
        SELECT display_name, wins, losses, regular_season_rank, final_rank
        FROM sleeper_standings
        WHERE league_id = '1240509989819273216'
        ORDER BY COALESCE(final_rank, 999)
        """
    ).fetchall()
season_2025_rows

[('nolmacdonald', 16, 12, 2, 1),
 ('ksavabi', 14, 14, 6, 2),
 ('nawfeastdallas', 15, 13, 5, 3),
 ('casitzmann', 20, 8, 1, 4),
 ('thatbolb', 16, 12, 4, 5),
 ('jwhitney0220', 16, 12, 3, 6),
 ('hyoga10', 11, 17, 9, None),
 ('aperry151', 11, 17, 8, None),
 ('Donkeysride', 12, 16, 7, None),
 ('ruhbberduhcky', 9, 19, 10, None)]

In this league's real 2025 season, the team with the best regular-season record finished **4th** -- the eventual champion had fewer wins and won it in the playoffs.

Final placement is deliberately conservative: it's computed only from winners-bracket matches that carry a `p` (placement) field via `resolve_final_ranks` -- a match's winner gets rank `p`, its loser `p + 1`. The losers bracket's own placement field is captured raw in `sleeper_playoff_matches` but never turned into a guessed `final_rank`, because its numbering convention couldn't be confirmed against real data -- rosters that only reached the losers bracket show `final_rank = NULL` rather than a number that might be wrong. (A Sleeper "Chopped" league has no playoff bracket at all; `is_chopped_league` and `resolve_chopped_final_ranks` handle that case from the elimination order instead.)

> **Why this matters:** notice the `with duckdb.connect(...) as conn:` block above closes `conn` again as soon as the query finishes, rather than keeping one connection open for the rest of the notebook. That's deliberate, not just tidiness -- a read-only connection left open would block the next `fetch_and_write_*` write cell below from opening its own connection to the same file. Reconnect fresh before every query in a notebook like this one, rather than reusing a connection across cells that also write.

## Weekly matchups

`fetch_and_write_matchups` fetches every week's roster-vs-roster scoring, for every season in the chain:

In [6]:
from nuclearff.sleeper import fetch_and_write_matchups

with SleeperClient(cache_dir="./demo/data/cache") as client:
    matchup_count = fetch_and_write_matchups(client, leagues, db_path, max_week=3)

print(f"Matchups: {matchup_count} roster-week row(s)")

Matchups: 180 roster-week row(s)


Each row in `sleeper_matchups` carries points, the matchup id pairing two rosters together, and JSON columns for `players`/`starters`/`players_points` -- a full weekly box score per roster. `max_week` caps how many weeks are fetched per season (default 18, a full regular + postseason); weeks that haven't happened yet return no data and are silently skipped, not treated as an error.

## Transaction history

`fetch_and_write_transactions` fetches every week's adds, drops, waivers, and trades, across the full history chain:

In [7]:
from nuclearff.sleeper import fetch_and_write_transactions

with SleeperClient(cache_dir="./demo/data/cache") as client:
    tx_count, tx_player_count = fetch_and_write_transactions(
        client, leagues, db_path, max_week=3
    )

print(f"Transactions: {tx_count}")
print(f"Add/drop rows: {tx_player_count}")

Transactions: 368
Add/drop rows: 546


Sleeper's own `type` and `status` fields are trusted and stored as-is, not re-derived:

In [8]:
with duckdb.connect(db_path, read_only=True) as conn:
    tx_type_counts = conn.execute(
        "SELECT type, COUNT(*) FROM sleeper_transactions GROUP BY type"
    ).fetchall()
tx_type_counts

[('trade', 5), ('waiver', 256), ('free_agent', 107)]

Two tables land in DuckDB: `sleeper_transactions` (one row per transaction -- `creator`/`roster_ids`/`consenter_ids` resolved to display names alongside the raw ids, timestamps parsed from Sleeper's epoch-millisecond format to real UTC `TIMESTAMP` values) and `sleeper_transaction_players` (`adds`/`drops` unnested to one row per player per direction).

## Roster composition

`fetch_and_write_roster_players` categorizes every roster's players by slot, per season:

In [9]:
from nuclearff.sleeper import fetch_and_write_roster_players

with SleeperClient(cache_dir="./demo/data/cache") as client:
    roster_player_count = fetch_and_write_roster_players(client, leagues, db_path)

print(f"Roster players: {roster_player_count}")

Roster players: 947


In [10]:
with duckdb.connect(db_path, read_only=True) as conn:
    slot_counts = conn.execute(
        "SELECT slot, COUNT(*) FROM sleeper_roster_players GROUP BY slot"
    ).fetchall()
slot_counts

[('bench', 358), ('reserve', 49), ('starter', 540)]

Each row is a `(league_id, season, roster_id, player_id)` with a `slot` of `starter`, `reserve` (IR), `taxi`, or `bench` -- join it to `sleeper_players` (below) for the player's name and position. Sleeper fills an empty starting slot with the literal string `"0"` before a draft; that filler is dropped rather than stored as a phantom player.

Every one of these writes replaces or merges its own table's rows (see `17_querying_and_provenance.ipynb` for the `merge_table` vs. `replace_table` distinction, and a real bug it fixed) -- a fresh run is a fresh full snapshot for that league, not an incremental append. There's no ordering requirement between the functions above; call the ones you need.

## Combining flags

Every fetch function above takes the same `leagues` chain, so the fastest real workflow is to just call every one you need back to back, as this notebook has been doing. There's no single "fetch everything" function in the library layer -- that convenience wrapper is `18_cli_reference.ipynb`'s `sleeper fetch-league --history --standings --matchups --transactions --roster-players` in one shell command instead.

## Discovering leagues from a username

Every function above needs an already-known `league_id`. If you only have a Sleeper username, resolve it first, the same way `03_sleeper_api.ipynb` did:

In [11]:
with SleeperClient() as client:
    user = client.get_user("nolmacdonald")
    disc_leagues = client.get_user_leagues(user["user_id"], 2026, sport="nfl")

print(f"{user['display_name']} ({user['user_id']})")
print(f"{len(disc_leagues)} leagues for 2026")
for league in disc_leagues[:3]:
    print(f"  {league['league_id']}  {league['name']}  status={league.get('status')}")

nolmacdonald (332632476830679040)
18 leagues for 2026
  1313346760332029952  NUCLEARFF DYNASTY  status=in_season
  1367225133634191360  NUCLEARFF REDRAFT  status=in_season
  1397668648528650240  Antares Sunday Operations  status=in_season


`client.get_user_drafts(user_id, season)` works the same way for drafts instead of leagues.

## The player map, and filtering server-side

`write_players_table` pulls Sleeper's full NFL player map into a `sleeper_players` DuckDB table -- this is the join target for every `player_id` column above:

In [12]:
from nuclearff.sleeper import write_players_table

with SleeperClient(cache_dir="./demo/data/cache") as client:
    players = client.get_players()

count = write_players_table(players, db_path)
print(f"Players: {count}")

Players: 12228


As covered in `03_sleeper_api.ipynb`, filter server-side with `get_players(position=..., active=...)` instead of fetching everything when you only need part of the map.

## Cross-referencing to nflverse

Sleeper player objects already carry several cross-platform IDs, including `gsis_id` -- nflverse's own primary key -- but not every player has one. `nuclearff.ids.crosswalk` fills the gap from nflverse's own ID crosswalk:

In [13]:
from nuclearff.ids import (
    ambiguous_sleeper_ids,
    read_sleeper_players,
    resolve_missing_gsis_ids,
    write_player_id_map,
)
from nuclearff.nflverse import load_ff_playerids

players_df = read_sleeper_players(db_path)
ff_ids = load_ff_playerids()

resolved = resolve_missing_gsis_ids(players_df, ff_ids)
n_written = write_player_id_map(resolved, db_path)

print(resolved["gsis_id_source"].value_counts())
print(f"Skipped {ambiguous_sleeper_ids(ff_ids).height} ambiguous crosswalk row(s).")

shape: (3, 2)
┌────────────────┬───────┐
│ gsis_id_source ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ null           ┆ 4809  │
│ sleeper        ┆ 3893  │
│ ff_playerids   ┆ 3526  │
└────────────────┴───────┘
Skipped 12 ambiguous crosswalk row(s).


`read_sleeper_players` requires `write_players_table` to have run first. The result -- written by `write_player_id_map` to a `player_id_map` table -- records each player's best-known `gsis_id` and where it came from: Sleeper's own field, or the crosswalk. A crosswalk row whose `sleeper_id` maps to more than one player is skipped rather than guessed (`ambiguous_sleeper_ids`); most of what's still unresolved is players missing from this year's crosswalk entirely, typically rookies.

## Querying what you've built

Every table above lives in one DuckDB file:

In [14]:
with duckdb.connect(db_path, read_only=True) as conn:
    conn.sql("SHOW TABLES").show()

┌─────────────────────────────┐
│            name             │
│           varchar           │
├─────────────────────────────┤
│ player_id_map               │
│ sleeper_league_configs      │
│ sleeper_leagues             │
│ sleeper_matchups            │
│ sleeper_players             │
│ sleeper_players_meta        │
│ sleeper_playoff_matches     │
│ sleeper_roster_players      │
│ sleeper_standings           │
│ sleeper_transaction_players │
│ sleeper_transactions        │
│ sleeper_user_avatars        │
└─────────────────────────────┘
            12 rows          



(the full set, once every function on this page has been called at least once -- your own database only has tables for what you've actually run). Every table is keyed by `league_id` (and usually `season`), so joining across them -- "which player did the eventual champion trade for" or "how many waiver claims did the team with the worst record make" -- is ordinary SQL, not custom Python per question. `17_querying_and_provenance.ipynb` covers this in more depth, alongside recording *how* a dataset was built.

## What's Next

With a real, persisted league dataset in place, `05_scoring_engine.ipynb` starts turning real NFL statistics into fantasy points under this league's own rules -- the foundation everything in Chapters 6 through 12 builds on.

## See Also

- `03_sleeper_api.ipynb` -- the client this notebook builds on.
- [Capturing and Storing a League](https://nolmacdonald.github.io/nuclearff/tutorial/04_capturing_a_league.html) -- the matching docs page.